# Nhóm 18 - Dữ liệu mô phỏng và thí nghiệm Learning-to-Rank

Notebook chạy xuyên suốt **không cần dữ liệu doanh nghiệp**. Toàn bộ tài sản, địa danh, mã hồ sơ, mô tả, giá và nhãn được sinh mới.
Các phân vị tham khảo dưới đây chỉ là thống kê tổng hợp đã khảo sát trước. Kết quả không chứng minh hiệu quả định giá thực tế.

Thứ tự: sinh tài sản → tạo nhãn → kiểm tra và xuất ba bảng → dựng cặp train/dev → baseline và LambdaRank trên dev.
Test T10 chỉ được sinh và kiểm tra cấu trúc; không đánh giá mô hình trong notebook này.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src/data.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import (
    FEATURES, DEFAULT_DATA, generate_assets, fit_text,
    generate_labels, validate_data, save_data, load_data, build_pairs,
)
from src.experiment import run_dev


## 1. Thiết kế đã chốt trước thí nghiệm

Phạm vi tham khảo: đất ở HCM không kèm mục đích khác; bộ mô phỏng dùng `SIM_HCM`, 10 xã giả, 5 đường mỗi xã và cùng đơn vị đ/m².

| Thuộc tính | P5 | P25 | P50 | P75 | P95 |
|---|---:|---:|---:|---:|---:|
| Diện tích tham khảo (m²) | 41 | 68,84 | 100 | 150 | 350 |
| KC dương tham khảo (m) | 30 | 80 | 150 | 290 | 650 |
| Ngõ khi KC=0 (m) | 5 | 7 | 8 | 14 | 25 |
| Ngõ khi KC>0 (m) | 1,66 | 3 | 4,5 | 7,5 | 13 |

Tỷ lệ KC=0 tham khảo: 6.576/12.364. Mặt tiền được sinh theo năm nhóm diện tích, ngõ theo nhóm KC.

**Giả định:** diện tích 20–600 m²; đuôi mặt tiền/ngõ từ 0,5×P5 đến 1,5×P95; chọn đều xã, đường và ngày trong từng cửa sổ;
16 mẫu mô tả tự viết. Đơn giá dùng công thức giả định và seed 30, chỉ phục vụ APE, không phải giá thật.

Giữ các seed 18–26 của bản thử; nhãn train/dev/test lần lượt dùng seed 27/28/29. Các mốc đầy đủ nằm trong `src/data.py` và manifest.


In [2]:
tstd, kho = generate_assets()

print("TSTĐ:", tstd.shape, "Kho:", kho.shape)
lich = tstd.groupby("tap", sort=False).agg(
    so_tstd=("ma_kho", "size"), tu_ngay=("ngay", "min"), den_ngay=("ngay", "max")
)
for cot in ["tu_ngay", "den_ngay"]:
    lich[cot] = lich[cot].dt.strftime("%d/%m/%Y")
lich


TSTĐ: (700, 15) Kho: (5000, 14)


,so_tstd,tu_ngay,den_ngay
tap,,,
train,400,01/01/2026,30/07/2026
dev,100,01/08/2026,30/09/2026
test,200,01/10/2026,31/10/2026


## 2. Nhãn mô phỏng, chất lượng dữ liệu và ba bảng xuất

Luật nhãn v1: cùng đường (trọng số 2), cùng nhóm KC (1), cosine văn bản (1), giống diện tích (1), giống mặt tiền (0,5),
giống ngõ (0,5), gần KC (0,5), độ mới (0,25), cộng Uniform(-0,5; 0,5), chọn ba ứng viên. Công thức đầy đủ trong README.

TF-IDF chỉ fit mô tả TSTĐ train. Điểm sinh nhãn không là đặc trưng. Trần sau bộ lọc cơ bản là 100% theo thiết kế nhãn.

Mỗi hồ sơ `gui` có STT 0 là TSTĐ và STT 1–3 là ba lựa chọn; **STT không phải nhãn y**. Nhãn học y=1 khi ứng viên nằm trong ba lựa chọn.
Bộ kiểm tra xác nhận mã duy nhất, số báo cáo, thời gian, cùng địa bàn/đơn vị, nhãn tồn tại và đủ ba lựa chọn.
Dữ liệu CSV có tên cột tiếng Việt tương ứng các trường gốc được dùng; không tái tạo toàn bộ cột quản trị không cần thiết.


In [3]:
vec = fit_text(tstd)
gui = generate_labels(tstd, kho, vec)
thong_ke_du_lieu = validate_data(tstd, kho, gui)
save_data(tstd, kho, gui, DEFAULT_DATA)

print("Số dòng tstd/kho/gui:", len(tstd), len(kho), len(gui))
print("Đã xuất CSV và manifest vào:", DEFAULT_DATA)
thong_ke_du_lieu


Số dòng tstd/kho/gui: 700 5000 2800
Đã xuất CSV và manifest vào: /private/var/folders/zn/nvvcq9255lvcphqr4_pfv6x00000gn/T/ml_project_clean_clone_edehqhrl/data/gia_lap


,so_tstd,so_cap,uv_it_nhat,uv_trung_vi,uv_nhieu_nhat
tap,,,,,
train,400,109487,250,272.0,307
dev,100,27546,254,271.5,318
test,200,56171,259,277.0,325


In [4]:
# Ví dụ hoàn toàn mô phỏng: một hồ sơ gồm TSTĐ và ba lựa chọn.
gui.head(4)


,SO_BAO_CAO,MA_KHO,STT
0,SIM_BC_TD_00001,SIM_TD_00001,0
1,SIM_BC_TD_00001,SIM_KHO_03860,1
2,SIM_BC_TD_00001,SIM_KHO_02617,2
3,SIM_BC_TD_00001,SIM_KHO_01629,3


## 3. Bảng cặp để huấn luyện và đánh giá

Chỉ 8 cột trong `FEATURES` là đầu vào. Ngoài nhãn và hai khóa, bảng cặp có thêm giá TSTĐ, giá ứng viên và tỷ lệ ngõ làm
metadata đánh giá/POC; các cột này không được truyền vào mô hình. Vì vậy bảng có 14 cột, thay vì 11 ở bản thử trước.
Giá TSTĐ không được dùng khi xếp hạng.


In [5]:
# Đọc lại ba bảng để kiểm tra quy trình xuất/nhập.
tstd, kho, gui = load_data(DEFAULT_DATA)
validate_data(tstd, kho, gui)
vec = fit_text(tstd)
train_pairs = build_pairs(tstd, kho, gui, vec, "train")
dev_pairs = build_pairs(tstd, kho, gui, vec, "dev")
COT_DAC_TRUNG = FEATURES.copy()

kiem_tra = []
for ten, bang in [("train", train_pairs), ("dev", dev_pairs)]:
    nhan = bang.groupby("so_bc")["y"].sum()
    kiem_tra.append({
        "tap": ten, "so_cap": len(bang), "so_cot": bang.shape[1],
        "so_nhom": len(nhan), "nhan_duong": int(bang.y.sum()),
        "nhom_sai_nhan": int(nhan.ne(3).sum()),
        "gia_tri_thieu": int(bang.isna().sum().sum()),
        "cap_lap": int(bang.duplicated(["so_bc", "ma_kho"]).sum()),
    })
pd.DataFrame(kiem_tra).set_index("tap")


,so_cap,so_cot,so_nhom,nhan_duong,nhom_sai_nhan,gia_tri_thieu,cap_lap
tap,,,,,,,
train,109487,14,400,1200,0,0,0
dev,27546,14,100,300,0,0,0


## 4. Thí nghiệm tiến độ trên dev

**Câu hỏi:** LambdaRank có cải thiện Hit@3/NDCG@3 so với luật đơn giản?
**Giả thuyết:** kết hợp 8 đặc trưng học luật có nhiễu tốt hơn thứ tự ưu tiên cố định.
**Tiêu chí đặt trước:** Hit@3 cao hơn luật đơn giản, NDCG@3 không thấp hơn. Chưa tuyên bố ý nghĩa thống kê.

Cấu hình cố định: 300 cây, learning rate 0,05, 15 lá, min_child_samples=20, seed 8; không tìm kiếm tham số, không early stopping.
Baseline: ngẫu nhiên seed 8; cùng đường → giống mô tả → mới hơn; lọc POC (cùng nhóm KC, ngõ 0,7–1,5×) rồi luật đơn giản.

Hit@3 có mẫu số 3×số TSTĐ. NDCG@3 giữ IDCG của ba nhãn ban đầu, kể cả sau lọc POC. APE chỉ tính khi chọn đủ ba ứng viên,
đơn giá hoàn toàn mô phỏng và chưa áp dụng tỷ lệ điều chỉnh. Protocol được ghi trước mỗi lần chạy. Không đánh giá test.


In [6]:
ket_qua_dev, _, feature_importance = run_dev(DEFAULT_DATA, ROOT / "reports/progress")
ket_qua_dev.round(3)


,phuong_an,n_tstd,Hit@3,NDCG@3,>=1_trung_%,du_3_%,tran_%,phu_3_uv_%,APE_trung_vi_%,n_APE
0,Ngẫu nhiên (seed 8),100,2.000,0.021,6.0,0.0,100.000,100.0,22.141,100
1,Luật đơn giản,100,43.333,0.469,83.0,4.0,100.000,100.0,12.181,100
2,Lọc POC + luật đơn giản,100,30.333,0.335,68.0,1.0,46.333,99.0,7.344,99
3,LambdaRank (8 đặc trưng),100,57.667,0.618,98.0,6.0,100.000,100.0,8.821,100


In [7]:
# Gain là mức đóng góp vào các phép chia cây, không phải quan hệ nhân quả.
feature_importance.round(3)


,dac_trung,gain
0,cung_duong,41906.289
7,ct_sim,7559.323
1,cung_nhom_kc,6960.957
2,dien_tich_lech,3833.192
4,ngo_lech,2009.057
6,do_cu_ngay,2002.193
3,mat_tien_lech,1711.945
5,kc_lech,1362.463


## 5. Giới hạn và việc tiếp theo

Nhãn được sinh từ chính các đặc trưng có nhiễu; kết quả thể hiện học lại quy tắc trong mô phỏng. Văn bản chỉ có 16 tổ hợp,
địa bàn chọn đều, chưa có lỗi nhập liệu được cài thêm. Giá không phải dữ liệu thị trường.

Tiếp theo tập trung ML: giải thích LambdaRank và group; phân tích lỗi từng query; ablation bỏ văn bản trên dev;
so sánh tác động của lọc POC; chốt cấu hình/tiêu chí trước khi đánh giá test T10 một lần cho báo cáo cuối.
Không sửa bộ sinh để tăng Hit@3. Mã nguồn và kết quả cần được commit/push trước khi nộp liên kết cho giảng viên.
